The Observer design pattern defines a one-to-many dependency between objects, where a change in one object (the subject or publisher) automatically notifies all its dependent objects (the observers or subscribers). 
This pattern is particularly useful in scenarios where an event in one part of a system needs to trigger updates in multiple other parts without tightly coupling them together.


In [1]:
class SmartLight:
    def turn_on(self):
        print("Light: Turned ON.")

class SecuritySystem:
    def sound_alarm(self):
        print("Security: ALARM TRIPPED!")

class MotionSensor:
    def __init__(self):
        self.light = SmartLight()
        self.security = SecuritySystem()

    def detect_motion(self):
        print("Sensor: Motion detected!")
        # The pain point: Manual calls
        self.light.turn_on()
        self.security.sound_alarm()

# Execution
sensor = MotionSensor()
sensor.detect_motion()


Sensor: Motion detected!
Light: Turned ON.
Security: ALARM TRIPPED!


### Above code using Observer

In [3]:
from abc import ABC, abstractmethod

# 1. The Interface: Any device that wants to react to motion must follow this rule
class MotionSubscriber(ABC):
    @abstractmethod
    def update(self):
        pass

# 2. The Subject: Only manages a list; doesn't know WHO is in it
class MotionSensor:
    def __init__(self):
        self._subscribers = []

    def attach(self, device: MotionSubscriber):
        self._subscribers.append(device)

    def detect_motion(self):
        print("\n[Sensor] Motion detected! Notifying subscribers...")
        for device in self._subscribers:
            device.update()

# 3. Concrete Observers: Devices can be added/removed without touching Sensor code
class SmartLight(MotionSubscriber):
    def update(self):
        print("SmartLight: Received signal -> Lights ON.")

class SecuritySystem(MotionSubscriber):
    def update(self):
        print("SecuritySystem: Received signal -> Monitoring active.")

class SmartCamera(MotionSubscriber): # Added easily without changing MotionSensor!
    def update(self):
        print("SmartCamera: Received signal -> Recording started.")

# --- Execution ---
sensor = MotionSensor()

# Plug-and-Play: Connect devices at runtime
sensor.attach(SmartLight())
sensor.attach(SecuritySystem())
sensor.attach(SmartCamera())

# One trigger, multiple independent reactions
sensor.detect_motion()



[Sensor] Motion detected! Notifying subscribers...
SmartLight: Received signal -> Lights ON.
SecuritySystem: Received signal -> Monitoring active.
SmartCamera: Received signal -> Recording started.


### Example 2

In [2]:
from abc import ABC, abstractmethod

# Observer Interface
class Observer(ABC):
    @abstractmethod
    def update(self, subject):
        pass

# Subject Class
class Subject:
    def __init__(self):
        self._observers = []

    def attach(self, observer):
        """Attach an observer to the subject."""
        if observer not in self._observers:
            self._observers.append(observer)
        else:
            print(f"Failed to add: {observer} (already attached)")

    def detach(self, observer):
        """Detach an observer from the subject."""
        try:
            self._observers.remove(observer)
        except ValueError:
            print(f"Failed to remove: {observer} (not found)")

    def notify(self):
        """Notify all observers about an event."""
        for observer in self._observers:
            observer.update(self)

# Concrete Subject
class WeatherStation(Subject):
    def __init__(self, location="Unknown"):
        super().__init__()
        self.location = location
        self._temperature = 0

    @property
    def temperature(self):
        return self._temperature

    @temperature.setter
    def temperature(self, new_temperature):
        """Set the temperature and notify observers of the change."""
        self._temperature = new_temperature
        self.notify()

# Concrete Observers
class PhoneDisplay(Observer):
    def update(self, subject):
        print(f"Phone Display ({subject.location}): Temperature updated to {subject.temperature}°C")

class TVDisplay(Observer):
    def update(self, subject):
        print(f"TV Display ({subject.location}): Weather updated - {subject.temperature}°C")

# Usage Example
if __name__ == "__main__":
    # Create the subject
    weather_station = WeatherStation(location="San Francisco")

    # Create the observers
    phone_display = PhoneDisplay()
    tv_display = TVDisplay()

    # Attach observers to the subject
    weather_station.attach(phone_display)
    weather_station.attach(tv_display)

    # Change the state of the subject, triggering notifications
    print("Setting temperature to 25°C:")
    weather_station.temperature = 25

    print("\nDetaching phone display and setting temperature to 30°C:")
    weather_station.detach(phone_display)
    weather_station.temperature = 30


Setting temperature to 25°C:
Phone Display (San Francisco): Temperature updated to 25°C
TV Display (San Francisco): Weather updated - 25°C

Detaching phone display and setting temperature to 30°C:
TV Display (San Francisco): Weather updated - 30°C
